# Simulated Quantum Annealing for ICP
+ This code simulates quantum annealing to solve the Iterative Closest Point (ICP) problem.

In [1]:
import numpy as np
import openjij as oj
import open3d as o3d
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Data generation

In [2]:
data_seed = 20251023

In [16]:
np.random.seed(data_seed)

X = np.random.randint(low=-3, high=3, size=(30, 3))

# X = np.array([
#     [0.0, 0.0],
#     [1.0, 0.0],          
#     [0.0, 1.0],
#     [1.0, 1.0],
#     [0.5, 0.5],
#     [0.5, 0.3],
#     [0.3, 0.5],
#     [0.7, 0.7],
#     [0.2, 0.8],
#     [0.8, 0.2]
# ])

t_true = np.array([0.5, 0.3, -0.5])
Y = X + t_true + 0.05 * np.random.normal(*X.shape)

In [17]:
# Y = Y[:10]

In [18]:
N = X.shape[0]
M = Y.shape[0]

In [33]:
penalty_i_must_one = 2000.0
penalty_j_must_one = 2000.0

In [34]:
# Q = {}
# for i in range(N):
#     for j in range(M):
#         idx = (i, j)
#         cost = np.sum((X[i] - Y[j])**2)
#         Q[idx, idx] = cost - penalty_i_must_one - penalty_j_must_one

#     for j1 in range(M):
#         for j2 in range(j1+1, M):
#             Q[(i, j1), (i, j2)] = penalty_j_must_one# + penalty_one

# for j in range(M):
#     for i1 in range(N):
#         for i2 in range(i1+1, N):
#             if ((i1, j), (i2, j)) in Q:
#                 Q[(i1, j), (i2, j)] += penalty_i_must_one# + penalty_one
#             else:
#                 Q[(i1, j), (i2, j)] = penalty_i_must_one


In [35]:
Q = {}
for i in range(N):
    for j in range(M):
        idx = (i, j)
        cost = np.sum((X[i] - Y[j])**2)
        Q[idx, idx] = cost - 2*(penalty_i_must_one + penalty_j_must_one)

    # i 固定：複数の j 禁止
    for j1 in range(M):
        for j2 in range(j1+1, M):
            Q[(i, j1), (i, j2)] = 2*penalty_j_must_one

# j 固定：複数の i 禁止
for j in range(M):
    for i1 in range(N):
        for i2 in range(i1+1, N):
            Q[(i1, j), (i2, j)] = 2*penalty_i_must_one

In [36]:
sampler = oj.SQASampler()
response = sampler.sample_qubo(Q, num_reads=100)

In [37]:
sum([Q[(i,i), (i,i)] for i in range(M)])

np.float64(-239746.0511964199)

In [38]:
response.first.energy, response.first.num_occurrences, response.first.sample

(np.float64(-239772.84794759483),
 np.int64(1),
 {(0, 0): np.int8(1),
  (0, 1): np.int8(0),
  (0, 2): np.int8(0),
  (0, 3): np.int8(0),
  (0, 4): np.int8(0),
  (0, 5): np.int8(0),
  (0, 6): np.int8(0),
  (0, 7): np.int8(0),
  (0, 8): np.int8(0),
  (0, 9): np.int8(0),
  (0, 10): np.int8(0),
  (0, 11): np.int8(0),
  (0, 12): np.int8(0),
  (0, 13): np.int8(0),
  (0, 14): np.int8(0),
  (0, 15): np.int8(0),
  (0, 16): np.int8(0),
  (0, 17): np.int8(0),
  (0, 18): np.int8(0),
  (0, 19): np.int8(0),
  (0, 20): np.int8(0),
  (0, 21): np.int8(0),
  (0, 22): np.int8(0),
  (0, 23): np.int8(0),
  (0, 24): np.int8(0),
  (0, 25): np.int8(0),
  (0, 26): np.int8(0),
  (0, 27): np.int8(0),
  (0, 28): np.int8(0),
  (0, 29): np.int8(0),
  (1, 0): np.int8(0),
  (1, 1): np.int8(0),
  (1, 2): np.int8(0),
  (1, 3): np.int8(0),
  (1, 4): np.int8(0),
  (1, 5): np.int8(0),
  (1, 6): np.int8(0),
  (1, 7): np.int8(0),
  (1, 8): np.int8(0),
  (1, 9): np.int8(0),
  (1, 10): np.int8(0),
  (1, 11): np.int8(0),
  (1, 

In [39]:
state = response.first.sample

In [40]:
[(i, j) for (i, j), bit in state.items() if bit == 1]

[(0, 0),
 (1, 21),
 (2, 2),
 (3, 24),
 (3, 28),
 (4, 9),
 (5, 3),
 (5, 5),
 (6, 6),
 (7, 4),
 (8, 8),
 (9, 14),
 (10, 10),
 (11, 11),
 (12, 1),
 (12, 12),
 (13, 13),
 (14, 14),
 (15, 15),
 (16, 16),
 (16, 27),
 (17, 17),
 (18, 23),
 (19, 7),
 (19, 19),
 (20, 22),
 (21, 29),
 (22, 6),
 (23, 23),
 (24, 20),
 (25, 25),
 (26, 26),
 (27, 2),
 (28, 18),
 (29, 29)]

In [28]:
state

{(0, 0): np.int8(1),
 (0, 1): np.int8(0),
 (0, 2): np.int8(0),
 (0, 3): np.int8(0),
 (0, 4): np.int8(0),
 (0, 5): np.int8(0),
 (0, 6): np.int8(0),
 (0, 7): np.int8(0),
 (0, 8): np.int8(0),
 (0, 9): np.int8(0),
 (1, 0): np.int8(0),
 (1, 1): np.int8(1),
 (1, 2): np.int8(0),
 (1, 3): np.int8(0),
 (1, 4): np.int8(0),
 (1, 5): np.int8(0),
 (1, 6): np.int8(0),
 (1, 7): np.int8(0),
 (1, 8): np.int8(0),
 (1, 9): np.int8(0),
 (2, 0): np.int8(0),
 (2, 1): np.int8(0),
 (2, 2): np.int8(1),
 (2, 3): np.int8(0),
 (2, 4): np.int8(0),
 (2, 5): np.int8(0),
 (2, 6): np.int8(0),
 (2, 7): np.int8(0),
 (2, 8): np.int8(0),
 (2, 9): np.int8(0),
 (3, 0): np.int8(0),
 (3, 1): np.int8(0),
 (3, 2): np.int8(0),
 (3, 3): np.int8(1),
 (3, 4): np.int8(1),
 (3, 5): np.int8(0),
 (3, 6): np.int8(0),
 (3, 7): np.int8(0),
 (3, 8): np.int8(0),
 (3, 9): np.int8(0),
 (4, 0): np.int8(0),
 (4, 1): np.int8(0),
 (4, 2): np.int8(0),
 (4, 3): np.int8(0),
 (4, 4): np.int8(0),
 (4, 5): np.int8(1),
 (4, 6): np.int8(0),
 (4, 7): np.i

In [9]:
Q

{((0, 0), (0, 0)): np.float64(0.5531790339495204),
 ((0, 1), (0, 1)): np.float64(2.7858251823948823),
 ((0, 2), (0, 2)): np.float64(2.3858251823948824),
 ((0, 0), (0, 1)): 10.0,
 ((0, 0), (0, 2)): 10.0,
 ((0, 1), (0, 2)): 10.0,
 ((1, 0), (1, 0)): np.float64(0.3205328855041583),
 ((1, 1), (1, 1)): np.float64(0.5531790339495204),
 ((1, 2), (1, 2)): np.float64(2.1531790339495203),
 ((1, 0), (1, 1)): 10.0,
 ((1, 0), (1, 2)): 10.0,
 ((1, 1), (1, 2)): 10.0,
 ((2, 0), (2, 0)): np.float64(0.7205328855041583),
 ((2, 1), (2, 1)): np.float64(2.95317903394952),
 ((2, 2), (2, 2)): np.float64(0.5531790339495205),
 ((2, 0), (2, 1)): 10.0,
 ((2, 0), (2, 2)): 10.0,
 ((2, 1), (2, 2)): 10.0}